# Candidate scoring prototype (Steps 2-3)

Scores HTR session candidates for the `A`/`X`-status rows (multiple
same-day candidates, no cross-day ambiguity) and the nearby-candidate
`-1`/`+1`/`?`-status rows (one or more previous-/next-day candidates) in
`session_date_status_1626_1630`. Combines entity-IDF overlap and dense
TF-IDF text similarity via `scripts.s4_candidate_scoring`.
Ranked suggestions only -- this does not auto-select a candidate.
See `docs/CANDIDATE_SCORING_AND_CONCORDANCE.md` for the design.

In [1]:
import sys
from collections import defaultdict
from pathlib import Path
from typing import Any

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / 'data_manifest.toml').exists()),
    Path.cwd(),
)
for import_root in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    if import_root.exists() and str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

import pandas as pd

from data_io import load, resolve
from build_alignment_new import calculate_idf_weights
from scripts.s4_corpus_paragraph_predictions import OVERLAP_DATASETS
from scripts.s4_paragraph_axis_baseline import build_axis_overlap
from scripts.s4_candidate_scoring import candidate_session_ids, is_nihil_actum, score_ledger_row, session_text

## 1. Load the ledger and filter to `A`/`X`/`-1`/`+1`/`?` rows

In [2]:
ledger = load('session_date_status_1626_1630')
candidate_rows = ledger.loc[ledger['status_code'].isin(['A', 'X', '-1', '+1', '?'])].sort_values(['inventory_id', 'enriched_date'])
print(f"{len(candidate_rows)} A/X/-1/+1/?-status rows")
candidate_rows[['session_date_key', 'inventory_id', 'enriched_date', 'status_code', 'trusted_session_ids', 'exact_date_session_ids', 'previous_day_session_ids', 'next_day_session_ids']]

610 A/X/-1/+1/?-status rows


,session_date_key,inventory_id,enriched_date,status_code,trusted_session_ids,exact_date_session_ids,previous_day_session_ids,next_day_session_ids
4,session-3185|1626-01-05,3185,1626-01-05,?,[],[],[session-3185-num-4],[session-3185-num-5]
8,session-3185|1626-01-09,3185,1626-01-09,?,[],[],[session-3185-num-7],[session-3185-num-8]
10,session-3185|1626-01-11,3185,1626-01-11,?,[],[],[session-3185-num-8],[session-3185-num-10]
15,session-3185|1626-01-16,3185,1626-01-16,-1,[],[],[session-3185-num-13],[]
17,session-3185|1626-01-18,3185,1626-01-18,+1,[],[],[],[session-3185-num-15]
...,...,...,...,...,...,...,...,...
2820,session-4562|1629-05-12,4562,1629-05-12,X,[],"[session-4562-num-249, session-4562-num-252, s...",[],[]
2821,session-4562|1629-05-13,4562,1629-05-13,?,[],[],"[session-4562-num-249, session-4562-num-252, s...",[session-4562-num-253]
2823,session-4562|1629-05-15,4562,1629-05-15,-1,[],[],[session-4562-num-253],[]
2829,session-4562|1629-05-21,4562,1629-05-21,+1,[],[],[],[session-4562-num-250]


## 2. Load supporting datasets and build lookups

Same machinery as `scripts/s4_session_date_mapping_predictions.py`: entity
overlap lookup + IDF weights from the place/org/person overlap workbooks,
flat resolution text grouped by session, enriched resolution text grouped
by date.

In [3]:
enriched_records = load('enriched_resolutions_1626_1630')
flat = load('resolutions_flat')
axis = load('paragraph_axis_1626_1630')

overlaps = [pd.read_excel(resolve(dataset)) for dataset in OVERLAP_DATASETS]
overlaps = [frame.rename(columns={'naam': 'name'}) if 'name' not in frame and 'naam' in frame else frame for frame in overlaps]
combined_overlap = pd.concat(overlaps, ignore_index=True)

overlap_lookup = build_axis_overlap(axis, combined_overlap)
idf_weights = calculate_idf_weights(combined_overlap)
print(f"{len(overlap_lookup)} (enriched_id, axis_id) overlap pairs; {len(idf_weights)} distinct entity names")

12541 (enriched_id, axis_id) overlap pairs; 912 distinct entity names


In [4]:
enriched_text_by_date: dict[str, str] = defaultdict(str)
enriched_by_date: dict[str, list[dict[str, Any]]] = defaultdict(list)
for record in enriched_records:
    enriched_by_date[str(record.get('date', ''))[:10]].append(record)
for date, records in enriched_by_date.items():
    records.sort(key=lambda item: item.get('resolution_index', 0))
    enriched_text_by_date[date] = ' '.join(str(item.get('text', '')) for item in records if item.get('text'))

flat_by_session: dict[str, list[dict[str, Any]]] = defaultdict(list)
for record in flat.to_dict(orient='records'):
    session_id = str(record.get('id', '')).split('-resolution-', 1)[0]
    flat_by_session[session_id].append(record)

axis_by_session: dict[str, list[dict[str, Any]]] = defaultdict(list)
for record in axis:
    session_id = str(record.get('flat_id', '')).split('-resolution-', 1)[0]
    axis_by_session[session_id].append(record)

print(f"{len(enriched_text_by_date)} dates with enriched text, {len(flat_by_session)} flat sessions, {len(axis_by_session)} axis sessions")

1595 dates with enriched text, 95245 flat sessions, 1316 axis sessions


## 3. Score every candidate for each row

In [5]:
scored_rows = []
for row in candidate_rows.to_dict(orient='records'):
    ranked = score_ledger_row(row, enriched_text_by_date, flat_by_session, axis_by_session, overlap_lookup, idf_weights)
    scored_rows.append({'row': row, 'ranked_candidates': ranked})

nihil_actum_count = 0
for entry in scored_rows:
    row = entry['row']
    top = entry['ranked_candidates'][0] if entry['ranked_candidates'] else None
    if top is None:
        if is_nihil_actum(enriched_text_by_date.get(str(row['enriched_date']), '')):
            nihil_actum_count += 1
            print(f"{row['session_date_key']} [{row['status_code']}] -> NIHIL_ACTUM, not scored")
        else:
            print(f"{row['session_date_key']} [{row['status_code']}] -> no candidates scored")
        continue
    flag = ' LOW_CONFIDENCE' if top['low_confidence'] else ''
    print(f"{row['session_date_key']} [{row['status_code']}] -> top candidate: {top['session_id']} (combined_score={top['combined_score']:.3f}, entity_overlap={top['entity_overlap_score']:.3f}){flag}")

print(f"\n{nihil_actum_count}/{len(scored_rows)} rows abstained as nihil actum")

session-3185|1626-01-05 [?] -> top candidate: session-3185-num-4 (combined_score=0.235, entity_overlap=0.000) LOW_CONFIDENCE
session-3185|1626-01-09 [?] -> top candidate: session-3185-num-7 (combined_score=0.254, entity_overlap=0.000) LOW_CONFIDENCE
session-3185|1626-01-11 [?] -> NIHIL_ACTUM, not scored
session-3185|1626-01-16 [-1] -> top candidate: session-3185-num-13 (combined_score=0.219, entity_overlap=0.000) LOW_CONFIDENCE
session-3185|1626-01-18 [+1] -> NIHIL_ACTUM, not scored
session-3185|1626-01-20 [?] -> top candidate: session-3185-num-15 (combined_score=0.242, entity_overlap=0.000) LOW_CONFIDENCE
session-3185|1626-01-22 [?] -> top candidate: session-3185-num-16 (combined_score=0.237, entity_overlap=0.000) LOW_CONFIDENCE
session-3185|1626-01-25 [?] -> NIHIL_ACTUM, not scored
session-3185|1626-01-29 [-1] -> top candidate: session-3185-num-22 (combined_score=0.234, entity_overlap=0.000) LOW_CONFIDENCE
session-3185|1626-02-02 [+1] -> top candidate: session-3185-num-23 (combined_s

## 4. Eyeball comparison: enriched text vs. each candidate's text and score

In [6]:
SNIPPET_CHARS = 600

for entry in scored_rows:
    row = entry['row']
    enriched_text = enriched_text_by_date.get(str(row['enriched_date']), '')
    print('=' * 100)
    print(f"{row['session_date_key']}  status={row['status_code']}  candidates={candidate_session_ids(row)}")
    print('-' * 100)
    if is_nihil_actum(enriched_text):
        print(f"NIHIL_ACTUM -- not scored, nothing to eyeball: {enriched_text[:SNIPPET_CHARS]!r}")
        print()
        continue
    print('ENRICHED TEXT:')
    print((enriched_text or '(none)')[:SNIPPET_CHARS])
    print()
    for rank, candidate in enumerate(entry['ranked_candidates'], start=1):
        flag = '  [LOW_CONFIDENCE: no shared entities -- verify manually, may be missing HTR content]' if candidate['low_confidence'] else ''
        print(f"  #{rank} {candidate['session_id']}  combined={candidate['combined_score']:.3f}  "
              f"entity_overlap={candidate['entity_overlap_score']:.3f}  dense_similarity={candidate['dense_similarity']:.3f}  "
              f"shared_entities={candidate['shared_entities']}{flag}")
        candidate_text = session_text(flat_by_session.get(candidate['session_id'], []))
        print(f"     {candidate_text[:SNIPPET_CHARS]}")
    print()

session-3185|1626-01-05  status=?  candidates=['session-3185-num-4', 'session-3185-num-5']
----------------------------------------------------------------------------------------------------
ENRICHED TEXT:
HHM zullen aan de Admiraliteiten schrijven dat zij geïnformeerd willen worden over de uitvoering van de resolutie d.d. 18 oktober 1625 inzake de verhoging van de licenten op de goederen die naar Frankrijk worden uitgevoerd en over de mogelijkheid eenzelfde verhoging
 toe te passen op de waren die over de Wezer en de
 Elbe naar omliggende gebieden gaan. Ordonnantie wordt gedepêcheerd van 83 gld. 16 st. voor een reis van Halewijn en Antwerpen naar
 Amsterdam en het
 Noorderkwartier. De Amsterdamse Admiraliteit schrijft d.d. Amsterdam 19 dec. 1625 naar aanleiding van het rekest van commissaris Co

  #1 session-3185-num-4  combined=0.235  entity_overlap=0.000  dense_similarity=0.235  shared_entities=[]  [LOW_CONFIDENCE: no shared entities -- verify manually, may be missing HTR content]


## 5. Manual verification

For each row above, confirm the top-ranked candidate looks correct by eye
against the enriched text. With 603 `?`/`-1`/`+1` rows (vs. 7 for `A`/`X`),
eyeball a stratified sample per status rather than every row, and check
whether the `entity_overlap_score` floor from the 7-row `A`/`X` set
(`MIN_CONFIDENT_ENTITY_OVERLAP = 0.0`) still separates correct from
incorrect top picks here. Record the outcome in
`docs/CANDIDATE_SCORING_AND_CONCORDANCE.md`.